In [0]:
# Initialize Spark session
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('Spark Playground').getOrCreate()

#Copy the starter code or load the file path available in the problem statement 

from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

employees_data = [
    (1,  "Alice",  1, 120000),
    (2,  "Bob",    1, 130000),
    (3,  "Carol",  1, 115000),
    (4,  "Dave",   1, 140000),
    (5,  "Eve",    1, 110000),
    (6,  "Frank",  2, 75000),
    (7,  "Grace",  2, 80000),
    (8,  "Hank",   2, 60000),
    (9,  "Ivy",    3, 55000),
    (10, "Jack",   3, 50000),
    (11, "Kate",   3, 45000),
    (12, "Liam",   4, 90000),
    (13, "Mia",    4, 85000),
    (14, "Noah",   4, 70000),
    (15, "Olivia", 4, 65000),
]
employees_columns = ["employee_id", "name", "department_id", "salary"]
employees = spark.createDataFrame(employees_data, employees_columns)


In [0]:
employees.show()

employees.write\
    .format("delta")\
    .mode("append")\
    .saveAsTable('employee')

In [0]:

departments_data = [
    (1, "Engineering", 700000),
    (2, "Marketing",   200000),
    (3, "HR",          150000),
    (4, "Sales",       300000),
]
departments_columns = ["department_id", "department_name", "budget"]
departments = spark.createDataFrame(departments_data, departments_columns)



In [0]:
departments.show()

departments.write\
    .format("delta")\
    .mode("append")\
    .saveAsTable("department")    


In [0]:
%sql
-- # Top earner in each department

-- Not a goog approach
-- SELECT sum(e.salary)
-- FROM employee e left join department d on e.department_id=d.department_id
-- group by d.department_id

with ranked_emp as (
    select e.name, d.department_name, e.salary,
    ROW_NUMBER() OVER( PARTITION BY d.department_id order by e.salary DESC) as rnk
    from employee e left join department d 
    on e.department_id=d.department_id
)
select name,department_name,salary from ranked_emp where rnk = 1;

In [0]:
%sql
-- Budget analysis with top earner

with ranked_emp as (
    select
    d.department_name as department_name,
    sum(e.salary) over(partition by d.department_id)as total_salary,
    d.budget as budget,
    round((budget/total_salary)*100,2) as utilisation_pct,
    case 
        when total_salary > budget then True 
        else false
    end as is_over_budget,
    e.name as top_earner_name,
    e.salary as top_earner_salary,
    ROW_NUMBER() OVER(PARTITION BY d.department_id order by e.salary desc)as rnk
    from employee e left join department d 
    on e.department_id=d.department_id 
)

select 
department_name,
total_salary,
budget,
is_over_budget,
utilisation_pct,
top_earner_name,
top_earner_salary
from ranked_emp
where rnk=1 




In [0]:
# Your logic goes here to create df_result
# temp_df = employees.join(departments, on='department_id', how='left')
# from pyspark.sql import functions as F
# agg_df = temp_df.groupBy('department_id','department_name', 'budget') \
#           .agg(F.sum('salary').alias('total_salary')) \
#           .withColumn('utilization_pct', F.round(F.col('total_salary')/F.col('budget')*100,2)) \
#           .withColumn('is_over_budget', F.col('total_salary') > F.col('budget'))

# from pyspark.sql.window import Window

# win_spec = Window.partitionBy('department_id').orderBy(F.desc('salary'))

# top_earners_df = temp_df.withColumn('rank', F.rank().over(win_spec)) \
#                   .filter(F.col('rank')==1) \
# .select('department_id', F.col('name').alias('top_earner_name'), F.col('salary').alias('top_earner_salary'))

# final_result = agg_df.join(top_earners_df, on='department_id', how='left')
# final_result = final_result.select('department_name', 'total_salary', 'budget', 'utilization_pct', 'is_over_budget', 'top_earner_name', 'top_earner_salary') \
#                 .orderBy('department_name')

employees.createOrReplaceTempView('employees')
departments.createOrReplaceTempView('departments')


query = """
with rank_emp as(
select 
d.department_name,
sum(e.salary) OVER(PARTITION BY d.department_id) as total_salary,
d.budget,
round((total_salary/budget)*100,2) as utilization_pct,
CASE
    WHEN total_salary > budget THEN True 
    ELSE False
    END as is_over_budget,
max(e.salary) OVER(PARTITION BY d.department_id) as top_earner_salary,
e.name as top_earner_name,
RANK() OVER(PARTITION BY d.department_id ORDER BY e.salary DESC) as rnk
from employees e
join departments d
on e.department_id = d.department_id
)
select 
department_name,
total_salary,
budget,
utilization_pct,
is_over_budget,
top_earner_name,
top_earner_salary

from rank_emp
where rnk=1

"""
final_result = spark.sql(query)
# Display the final DataFrame using the display() function.
display(final_result)